In [ ]:
path_to_dataset = "" #FIXME: Add path to dataset here

In [ ]:
import numpy as np
import csv
import os
import cv2
import itertools

import sys
sys.path.append('..')

from Pipeline import Pipeline as pl

In [ ]:
def training_border_classifier(csv_filepath, dataset_base_path, bw_range, th_range, rat_range):
    """
    Evaluates hyperparameters using dynamic ranges and a CSV dataset index.

    Ranges should be provided as tuples: (start, stop, step)
    """
    # 1. Generate lists dynamically
    band_widths_to_test = np.arange(bw_range[0], bw_range[1] + (bw_range[2]/2), bw_range[2], dtype=int)
    thresholds_to_test = np.arange(th_range[0], th_range[1] + (th_range[2]/2), th_range[2], dtype=int)
    ratios_to_test = np.arange(rat_range[0], rat_range[1] + (rat_range[2]/2), rat_range[2])
    
    print(f"Testing {len(band_widths_to_test)} bandwidths, {len(thresholds_to_test)} thresholds, and {len(ratios_to_test)} ratios.")
    print(f"Total combinations to test: {len(band_widths_to_test) * len(thresholds_to_test) * len(ratios_to_test)}\n")

    print("Parsing CSV and loading raw images into memory (this takes a moment)...")
    border_imgs = []
    no_border_imgs = []
    missing_files = 0

    # 2. Read from CSV and construct paths
    with open(csv_filepath, mode='r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            filename = row['image_filename'].strip()
            status = row['border_status'].strip()
            
            # Route by the first letter of the filename (e.g., 'a' -> 'A')
            subfolder = filename[0].upper()
            full_img_path = os.path.join(dataset_base_path, subfolder, filename)
            
            img = cv2.imread(full_img_path, cv2.IMREAD_GRAYSCALE)
            
            if img is not None:
                if status == 'border':
                    border_imgs.append(img)
                else:
                    no_border_imgs.append(img)
            else:
                missing_files += 1


    print(f"Loaded {len(border_imgs)} border images and {len(no_border_imgs)} no-border images.")
    if missing_files > 0:
        print(f"Warning: {missing_files} files could not be found/read from their expected paths.\n")

    results = []

    # 3. The 3D Grid Search Loop
    for bw in band_widths_to_test:
        print(f"Calculating for Bandwidth: {bw} pixels...")
        
        border_hists = [pl.get_perimeter_histogram(img, bw) for img in border_imgs]
        no_border_hists = [pl.get_perimeter_histogram(img, bw) for img in no_border_imgs]

        for threshold, ratio in itertools.product(thresholds_to_test, ratios_to_test):
            
            # true_positives: Border images correctly identified as having a border
            true_positives = sum(
                1 for hist in border_hists 
                if pl.border_classification(hist, threshold, ratio)
            )
                    
            # false_positives: No-border images incorrectly identified as having a border
            false_positives = sum(
                1 for hist in no_border_hists 
                if pl.border_classification(hist, threshold, ratio)
            )

            # False Negatives: The ones we NEEDED to keep, but accidentally rejected.
            # You want this to be 0.
            false_negatives = len(border_imgs) - true_positives

            recall = true_positives / len(border_imgs) if len(border_imgs) > 0 else 0

            results.append({
                'bandwidth': bw,
                'threshold': int(threshold),
                'ratio': ratio,
                'recall': recall,
                'fp_count': false_positives,
                'fn_count': false_negatives
            })

    # 4. Sort results according to the new hierarchy
    # reverse=True means positive values sort Highest->Lowest, negative values sort Lowest->Highest
    results.sort(
        key=lambda x: (
            x['recall'],       # 1. Recall (Highest first)
            -x['fp_count'],    # 2. False Positives (Lowest first)
            x['bandwidth'],    # 3. Bandwidth (Highest first)
            x['threshold'],    # 4. Threshold (Highest first)
            -x['ratio']        # 5. Ratio (Lowest first)
        ), 
        reverse=True
    )

    print("\nGrid search completed. Results sorted by recall, false positives, bandwidth, threshold, and ratio.")
    return results

In [ ]:
bw_range = (5, 25, 5)     
th_range = (0, 100, 2)    
rat_range = (0, 1, 0.02) 

results = training_border_classifier(
    csv_filepath='../Training_data/border_classification_dataset.csv', 
    dataset_base_path=path_to_dataset, 
    bw_range=bw_range, 
    th_range=th_range, 
    rat_range=rat_range
)

In [ ]:
print("Best result:")
for result in results[:1]:  # Display only the best result
    for key, value in result.items():
        print(f"  {key}: {value}")
    print(f'\n')

In [ ]:
output_csv_path = 'border_classification_training_results.csv'

print(f"\nSaving {len(results)} results to {output_csv_path}")
with open(output_csv_path, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['bandwidth', 'threshold', 'ratio', 'recall', 'fn_count', 'fp_count'])
    writer.writeheader()
    writer.writerows(results)